# Drug Review Insights & Summarization Tool: Data Ingestion and Preprocessing

**Dataset:** UCI ML Drug Review Dataset (Drugs.com) — 53,800 patient reviews covering 2,637 unique drugs across 708 conditions, with 10-star ratings, free-text reviews, and helpfulness votes spanning 2008–2017.

**Kaggle:** https://www.kaggle.com/datasets/jessicali9530/kuc-hackathon-winter-2018 

**UCI Repository:** https://archive.ics.uci.edu/dataset/461/drug+review+dataset+druglib+com 

## Data & Preprocessing

This notebook prepares the Drug Review dataset for downstream sentiment analysis, visualization, and GenAI summarization. The cleaned output is designed to be used by the main project pipeline and later modules.

Notebook: `preprocessing/drug_review_preprocessing.ipynb`

This notebook:
- Loads the Drug Review dataset
- Checks shape, columns, data types, missing values, and duplicates
- Cleans key fields such as `drugName`, `condition`, `review`, `rating`, `date`, and `usefulCount`
- Converts dates and numeric fields to the correct data types
- Creates preprocessing features such as `review_length`, `review_word_count`, `review_year`, and `rating_sentiment`
- Generates summary statistics and distribution plots
- Exports a cleaned dataset for downstream GenAI summarization and visualization

Output files:
- `preprocessing/cleaned_drug_reviews.csv`
- `preprocessing/drug_review_profile.csv`
- `preprocessing/figures/`

# Import Packages 

In [ ]:
import kagglehub
import os   
import pandas as pd
import numpy as np

/Users/nancywalker/Documents/University_SanDiego/ADS-507PracticalDataEngineering/ADS-507-Final-Team-Project/.conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Install ucimlrepo package if not already installed 
# !pip install ucimlrepo
from ucimlrepo import fetch_ucirepo 

## Load and Ingest Datasets 

- Read the UCI/Kaggle CSV files into pandas.
- Confirm row counts, columns, data types, and missing values.

In [3]:
# Download latest version
path = kagglehub.dataset_download("jessicali9530/kuc-hackathon-winter-2018")

print("Path to dataset files:", path)

Path to dataset files: /Users/nancywalker/.cache/kagglehub/datasets/jessicali9530/kuc-hackathon-winter-2018/versions/2


In [4]:
# Check that datasets are present
print("Files in dataset directory:", os.listdir(path))

# Load the datasets
train_df = pd.read_csv(
    os.path.join(path, "drugsComTrain_raw.csv")
)
test_df = pd.read_csv(
    os.path.join(path, "drugsComTest_raw.csv")
)

print("Train dataset shape:", train_df.shape)
print("Test dataset shape:", test_df.shape)

Files in dataset directory: ['drugsComTrain_raw.csv', 'drugsComTest_raw.csv']
Train dataset shape: (161297, 7)
Test dataset shape: (53766, 7)


This project is already split into a traing and testing dataset. To create our own split parameters after merging with other datasets will combine data then split. All preprocessingsteps that learns from the data (TF-IDF, scaling, embeddings normalization, SMOTE, etc.) are to be performed after splitting to avoid data leakage. 

In [5]:
# Save datasets to raw data directory for preprocessing
train_df.to_csv("../data/raw/drugsComTrain_raw.csv", index=False)
test_df.to_csv("../data/raw/drugsComTest_raw.csv", index=False)

In [6]:
# Combine original train and test
kaggle_full_df = pd.concat([train_df, test_df], ignore_index=True)
print("Full dataset shape:", kaggle_full_df.shape)

Full dataset shape: (215063, 7)


In [7]:
# Save combined dataset for preprocessing
kaggle_full_df.to_csv(
    "../data/processed/kaggle_combined.csv",
    index=False
)

In [8]:
# Check datatypes and missing values
print("Data types:\n", kaggle_full_df.dtypes)
print("\nMissing values:\n", kaggle_full_df.isnull().sum())

Data types:
 uniqueID        int64
drugName       object
condition      object
review         object
rating          int64
date           object
usefulCount     int64
dtype: object

Missing values:
 uniqueID          0
drugName          0
condition      1194
review            0
rating            0
date              0
usefulCount       0
dtype: int64


In [9]:
# View the percentage of missing values in each column
missing_percent = (kaggle_full_df.isnull().sum() / len(kaggle_full_df)) * 100
print("\nPercentage of missing values:\n", missing_percent)


Percentage of missing values:
 uniqueID       0.000000
drugName       0.000000
condition      0.555186
review         0.000000
rating         0.000000
date           0.000000
usefulCount    0.000000
dtype: float64


Important columns for text analysis have zero missing values. For example, review and rating. 

Condition has 0.56% columns with missing values. Since this column is not the target of analysis, ,issing rows can be filled with Unknown. 

In [10]:
# Fill missing condition values with "Unknown"
kaggle_full_df["condition"] = kaggle_full_df["condition"].fillna("Unknown")

# Check misisng value counts 
print("\nMissing values:\n", kaggle_full_df.isnull().sum())


Missing values:
 uniqueID       0
drugName       0
condition      0
review         0
rating         0
date           0
usefulCount    0
dtype: int64


In [11]:
kaggle_full_df.head()

,uniqueID,drugName,condition,review,rating,date,usefulCount
0,206461,Valsartan,Left Ventricular Dysfunction,"""It has no side effect, I take it in combinati...",9,20-May-12,27
1,95260,Guanfacine,ADHD,"""My son is halfway through his fourth week of ...",8,27-Apr-10,192
2,92703,Lybrel,Birth Control,"""I used to take another oral contraceptive, wh...",5,14-Dec-09,17
3,138000,Ortho Evra,Birth Control,"""This is my first time using any form of birth...",8,3-Nov-15,10
4,35696,Buprenorphine / naloxone,Opiate Dependence,"""Suboxone has completely turned my life around...",9,27-Nov-16,37


In [12]:
# Save cleaned dataset for preprocessing
kaggle_full_df.to_csv("../data/processed/kaggle_full_cleaned.csv", index=False)

- Read the UCI/Druglib CSV files into pandas.
- Confirm row counts, columns, data types, and missing values.

In [13]:
# fetch dataset 
drug_reviews_druglib_com = fetch_ucirepo(id=461) 
  
# data (as pandas dataframes) 
X = drug_reviews_druglib_com.data.features 
y = drug_reviews_druglib_com.data.targets 
  
# metadata 
print(drug_reviews_druglib_com.metadata) 
  
# variable information 
print(drug_reviews_druglib_com.variables) 

{'uci_id': 461, 'name': 'Drug Reviews (Druglib.com)', 'repository_url': 'https://archive.ics.uci.edu/dataset/461/drug+review+dataset+druglib+com', 'data_url': 'https://archive.ics.uci.edu/static/public/461/data.csv', 'abstract': 'The dataset provides patient reviews on specific drugs along with related conditions. Reviews and ratings are grouped into reports on the three aspects benefits, side effects and overall comment.', 'area': 'Health and Medicine', 'tasks': ['Classification', 'Regression', 'Clustering'], 'characteristics': ['Multivariate', 'Text'], 'num_instances': 4143, 'num_features': 8, 'feature_types': ['Integer'], 'demographics': [], 'target_col': None, 'index_col': ['reviewID'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2018, 'last_updated': 'Wed Apr 03 2024', 'dataset_doi': '10.24432/C55G6J', 'creators': ['Surya Kallumadi', 'Felix Grer'], 'intro_paper': {'ID': 457, 'type': 'NATIVE', 'title': 'Aspect-Based Sentiment Analysis of D

The UCI Drug Review dataset contains no missing values. 

In [14]:
# Convert UCI features into dataframe
druglib_df = X.copy()

Sometimes in the UCI dataset:

rating may already exist in X
and y could instead contain effectiveness

So before adding rating = y, run:

In [15]:
print(X.columns)
print(type(y))
print(y)

Index(['urlDrugName', 'rating', 'effectiveness', 'sideEffects', 'condition',
       'benefitsReview', 'sideEffectsReview', 'commentsReview'],
      dtype='object')
<class 'NoneType'>
None


All variable are already inside of X 

In [16]:
# Save Raw data after download/load
druglib_df.to_csv("../data/raw/druglib_df.csv", index=False)

In [17]:
# Preview
druglib_df.head()

,urlDrugName,rating,effectiveness,sideEffects,condition,benefitsReview,sideEffectsReview,commentsReview
0,enalapril,4,Highly Effective,Mild Side Effects,management of congestive heart failure,slowed the progression of left ventricular dys...,"cough, hypotension , proteinuria, impotence , ...","monitor blood pressure , weight and asses for ..."
1,ortho-tri-cyclen,1,Highly Effective,Severe Side Effects,birth prevention,Although this type of birth control has more c...,"Heavy Cycle, Cramps, Hot Flashes, Fatigue, Lon...","I Hate This Birth Control, I Would Not Suggest..."
2,ponstel,10,Highly Effective,No Side Effects,menstrual cramps,I was used to having cramps so badly that they...,Heavier bleeding and clotting than normal.,I took 2 pills at the onset of my menstrual cr...
3,prilosec,3,Marginally Effective,Mild Side Effects,acid reflux,The acid reflux went away for a few months aft...,"Constipation, dry mouth and some mild dizzines...",I was given Prilosec prescription at a dose of...
4,lyrica,2,Marginally Effective,Severe Side Effects,fibromyalgia,I think that the Lyrica was starting to help w...,I felt extremely drugged and dopey. Could not...,See above


### Create a Unified Schema

In [18]:
# Rename Columns for consistency
druglib_df = druglib_df.rename(columns={
    "urlDrugName": "drugName"
})

# Combine the three review text columns 
druglib_df["review"] = (
    druglib_df["benefitsReview"].fillna('') + " " +
    druglib_df["sideEffectsReview"].fillna('') + " " +
    druglib_df["commentsReview"].fillna('')
)

In [19]:
# Druglib subset with consistent columns for merging
druglib_subset = druglib_df[
    ["drugName", "condition", "review", "rating", "effectiveness", "sideEffects"]
].copy()

druglib_subset["date"] = None
druglib_subset["usefulCount"] = None
druglib_subset["source"] = "UCI_Druglib"

# Kaggle subset
kaggle_subset = kaggle_full_df[
    ["drugName", "condition", "review", "rating", "date", "usefulCount"]
].copy()

kaggle_subset["effectiveness"] = None
kaggle_subset["sideEffects"] = None
kaggle_subset["source"] = "Kaggle_DrugsCom"

In [20]:
# Combine the two datasets
combined_df = pd.concat(
    [kaggle_subset, druglib_subset],
    ignore_index=True
)

In [21]:
# Save combined dataset for preprocessing
combined_df.to_csv(
    "../data/processed/kaggle_druglib_combined.csv",
    index=False
)

## Clean The Data 

- Handle missing values in condition, review, rating, date, usefulCount.
- Remove duplicates.
- Clean review text lightly, but do not over-clean because the GenAI model needs readable patient language.
- Standardize column names.
- Convert date to datetime.
- Make sure ratings are numeric.

In [22]:
combined_df.head()

,drugName,condition,review,rating,date,usefulCount,effectiveness,sideEffects,source
0,Valsartan,Left Ventricular Dysfunction,"""It has no side effect, I take it in combinati...",9,20-May-12,27,None,None,Kaggle_DrugsCom
1,Guanfacine,ADHD,"""My son is halfway through his fourth week of ...",8,27-Apr-10,192,None,None,Kaggle_DrugsCom
2,Lybrel,Birth Control,"""I used to take another oral contraceptive, wh...",5,14-Dec-09,17,None,None,Kaggle_DrugsCom
3,Ortho Evra,Birth Control,"""This is my first time using any form of birth...",8,3-Nov-15,10,None,None,Kaggle_DrugsCom
4,Buprenorphine / naloxone,Opiate Dependence,"""Suboxone has completely turned my life around...",9,27-Nov-16,37,None,None,Kaggle_DrugsCom


Combined schema is clean and readable containing: 

    - structured metadata (drugName, condition)
    - patient-generated text (review)
    - quantitative sentiment proxy (rating)
    - engagement metric (usefulCount)
    - clinical descriptors (effectiveness, sideEffects)
    - source tracking (source)

Missing or "None" values in the sideEffects and effectiveness fields due to structural missing. Leave features alone.  

In [23]:
# Copy combined dataset for preprocessing
cleaned_df = combined_df.copy()

# Standardize column names
cleaned_df["drugName"] = cleaned_df["drugName"].str.strip()
cleaned_df["condition"] = cleaned_df["condition"].str.strip()

In [24]:
# Convert ratings to numeric    
cleaned_df["rating"] = pd.to_numeric(
    cleaned_df["rating"], 
    errors="coerce"
)

# Convert date columns to datetime if they exist
cleaned_df["date"] = pd.to_datetime(
    cleaned_df["date"],
    format="%d-%b-%y",
    errors="coerce"
)

In [25]:
# Light review cleaning
cleaned_df["review"] = (
    cleaned_df["review"]
    .astype(str)
    .str.strip() # remove leading/trailing spaces
    .str.replace(r"\s+", " ", regex=True) # normalize extra whitespace
)

In [26]:
# Save cleaned dataset for preprocessing
cleaned_df.to_csv(
    "../data/processed/cleaned_combined.csv",
    index=False
)

## Create Useful Features 

- Review length
- Sentiment label from rating, such as:
    - 1–4 = negative
    - 5–6 = neutral
    - 7–10 = positive
- Year from date
- Possibly helpfulness-weighted rating

Sentiment analysis: What elements of a review make it more helpful to others? Which patients tend to have more negative reviews? Can you determine if a review is positive, neutral, or negative?


In [27]:
# Create review length feature
cleaned_df["review_length"] = (
    cleaned_df["review"]
    .str.split()
    .str.len()
)

In [28]:
# Create Sentiment Lable based on rating
def label_sentiment(rating):
    if rating <= 4:
        return "negative"
    elif rating <= 6:
        return "neutral"
    else:
        return "positive"

cleaned_df["sentiment"] = (
    cleaned_df["rating"]
    .apply(label_sentiment)
)

In [29]:
# Create year from date feature 
cleaned_df["review_year"] = cleaned_df["date"].dt.year

In [30]:
# Convert ratings to numeric and fill missing values with the median rating
cleaned_df["rating"] = pd.to_numeric(
    cleaned_df["rating"],
    errors="coerce"
)

# Convert usefulCount to numeric and fill missing values with 0
cleaned_df["usefulCount"] = pd.to_numeric(
    cleaned_df["usefulCount"],
    errors="coerce"
).fillna(0)

# Create helpfulness-weighted rating 
cleaned_df["weighted_rating"] = (
    cleaned_df["rating"] *
    np.log1p(cleaned_df["usefulCount"])
)

In [31]:
# Save cleaned dataset for preprocessing
cleaned_df.to_csv("../data/processed/combined_reviews_cleaned.csv", index=False)

## Output a structured data profile

- Summary statistics
- Missing value table
- Rating distribution
- Top drugs
- Top conditions
- Review length distribution
- Positive/negative review counts by condition or drug

In [32]:
# Create summary profile
summary_profile = pd.DataFrame({
    "metric": [
        "total_rows",
        "total_columns",
        "duplicate_rows",
        "unique_drugs",
        "unique_conditions",
        "average_rating",
        "median_rating",
        "average_review_length"
    ],
    "value": [
        cleaned_df.shape[0],
        cleaned_df.shape[1],
        cleaned_df.duplicated().sum(),
        cleaned_df["drugName"].nunique(),
        cleaned_df["condition"].nunique(),
        cleaned_df["rating"].mean(),
        cleaned_df["rating"].median(),
        cleaned_df["review_length"].mean()
    ]
})

summary_profile

,metric,value
0,total_rows,219206.000000
1,total_columns,13.000000
2,duplicate_rows,52.000000
3,unique_drugs,4211.000000
4,unique_conditions,2724.000000
5,average_rating,6.989184
6,median_rating,8.000000
7,average_review_length,85.335698


The combiend dataset is fairly large with 219,206 reviews. This will be helpful for NLP modeling and GenAI summarization. 

The dataset also contains a wide variety of medications (4,211 unique drugs) and medical conditions (2,724 unique conditions). This supports a broad sentiment and healthcare analysis 

The median rating is 8, which is higher than the midpoint. Thefore reviews may skew positive. 

The average length of a review is about 85 words, indicating that patients provied detaield narratives. 

Only 52 duplicate rows were identified suggesting that the data source is relativly clean. 



In [33]:
# Missing Values Table 
missing_values = pd.DataFrame({
    "column": cleaned_df.columns,
    "missing_count": cleaned_df.isnull().sum().values,
    "missing_percent": (cleaned_df.isnull().mean().values * 100).round(2)
})

missing_values

,column,missing_count,missing_percent
0,drugName,0,0.00
1,condition,1,0.00
2,review,0,0.00
3,rating,0,0.00
4,date,4143,1.89
5,usefulCount,0,0.00
6,effectiveness,215063,98.11
7,sideEffects,215063,98.11
8,source,0,0.00
9,review_length,0,0.00


Misisng values look good and make sense given the merged dataset structure. Missingness is structural not problematic.

effectiveness and sideEffects are mostly missing after merging. The Druglib dataset is likely much smaller than the Kaggle dataset. 

In [34]:
# Fill missing condition values with "Unknown"
cleaned_df["condition"] = cleaned_df["condition"].fillna("Unknown")

In [35]:
# Rating Distribution Table
rating_distribution = (
    cleaned_df["rating"]
    .value_counts()
    .sort_index()
    .reset_index()
)

rating_distribution.columns = ["rating", "count"]
rating_distribution

,rating,count
0,1,29338
1,2,9401
2,3,8913
3,4,6822
4,5,10949
5,6,8677
6,7,13018
7,8,25794
8,9,37321
9,10,68973


Ratings of 9 and 10 dominate the dataset.

Low rating of 1- 4 are less common 

Patient reviews are generally more positive, users may be more likely to leave a review when medication works well. 

In [36]:
# Top 10 Drugs by Average Rating
top_drugs = (
    cleaned_df.groupby("drugName")["rating"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

top_drugs.columns = ["drugName", "average_rating"]
top_drugs

,drugName,average_rating
0,A + D Cracked Skin Relief,10.0
1,Ovace Plus,10.0
2,Capmist DM,10.0
3,Palladone,10.0
4,PEG-3350 with Electolytes,10.0
5,Oxytocin,10.0
6,Oxymetholone,10.0
7,Oxyfast,10.0
8,Oxy-10,10.0
9,Ovidrel,10.0


In [37]:
# Sentiment Counts 
sentiment_counts = (
    cleaned_df["sentiment"]
    .value_counts()
    .reset_index()
)

sentiment_counts.columns = ["sentiment", "count"]
sentiment_counts

,sentiment,count
0,positive,145106
1,negative,54474
2,neutral,19626


Positive reviews dominate the dataset, suggesting patients are more likely to submit favorable experiences.

Neutral reviews are the smallest category, which is common in sentiment datasets because opinions tend to skew strongly positive or negative.

The class imbalance should be considered during modeling because classifiers may become biased toward predicting the positive class.

In [38]:
# Sentiment by condition
sentiment_by_condition = (
    cleaned_df.groupby(["condition", "sentiment"])
    .size()
    .reset_index(name="count")
    .sort_values(["condition", "count"], ascending=[True, False])
)

sentiment_by_condition.head()

,condition,sentiment,count
2,0</span> users found this comment helpful.,positive,67
0,0</span> users found this comment helpful.,negative,44
1,0</span> users found this comment helpful.,neutral,17
3,100</span> users found this comment helpful.,positive,1
4,105</span> users found this comment helpful.,positive,1


Condition column contains some corrupted HTML/text artifacts from the Druglib dataset.

These are not real medical conditions 

In [39]:
# Check condition corrupt values
cleaned_df[
    cleaned_df["condition"]
    .str.contains("users found this comment helpful", na=False)
].shape

(1171, 13)

About 1,1771 rows contain corrupted condition values, which is only abotu 0.53% of the dataset. Replacing values with Unknown is acceptible. 

In [40]:
cleaned_df["condition"] = cleaned_df["condition"].replace(
    r".*users found this comment helpful.*",
    "Unknown",
    regex=True
)

In [41]:
cleaned_df["condition"].value_counts().head()

condition
Birth Control    38436
Depression       12164
Pain              8245
Anxiety           7812
Acne              7435
Name: count, dtype: int64

Top conditions now make sense. 

In [42]:
# Sentiment by drug
sentiment_by_drug = (
    cleaned_df.groupby(["drugName", "sentiment"])
    .size()
    .reset_index(name="count")
    .sort_values(["drugName", "count"], ascending=[True, False])
)

sentiment_by_drug.head()

,drugName,sentiment,count
0,A + D Cracked Skin Relief,positive,1
1,A / B Otic,positive,2
4,Abacavir / dolutegravir / lamivudine,positive,57
2,Abacavir / dolutegravir / lamivudine,negative,7
3,Abacavir / dolutegravir / lamivudine,neutral,6


sentiment grouping is working properly

## Save cleaned outputs

- cleaned_drug_reviews.csv (the actual cleaned dataset)
- drug_review_profile.csv (Summary/statistics about the datset)

In [43]:
# Save final cleaned dataset and summary profile
cleaned_df.to_csv(
    "../data/final/cleaned_drug_reviews.csv",
    index=False
)

# Save summary profile
summary_profile.to_csv(
    "../data/final/drug_review_profile.csv",
    index=False
)